In [1]:
# Setup the Jupyter version of Dash
from dash import Dash  # Import JupyterDash for running Dash in Jupyter Notebook

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl  # Import for geolocation mapping
from dash import dcc  # Import Dash core components for interactivity
from dash import html  # Import Dash HTML components for layout
import plotly.express as px  # Import for creating interactive plots
from dash import dash_table  # Import for creating data tables
from dash.dependencies import Input, Output, State  # Import for callback dependencies
import base64  # Import to encode the logo image

# Configure OS routines
import os  # Import for operating system-related functions (e.g., file path handling)

# Configure the plotting routines
import numpy as np  # Import for numerical operations
import pandas as pd  # Import for data manipulation and analysis
import matplotlib.pyplot as plt  # Import for basic plotting (though not used here directly)


#### FIX ME #####
# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from pymongo import MongoClient
from animal_shelter_enhanced import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# FIX ME update with your username and password and CRUD Python module name


# Connect to database via CRUD Module
client = MongoClient("mongodb://127.0.0.1:27017/")
collection = client["CS499_AnimalRescue_Performance"]["animals"]
db = AnimalShelter(collection=collection)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))  # Convert MongoDB data to a Pandas DataFrame using all records

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invalid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will return a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'], inplace=True)  # Remove '_id' column to avoid data table crash, modify dataframe in place

## Debug
# print(len(df.to_dict(orient='records')))  # Uncomment to debug: print number of records
# print(df.columns)  # Uncomment to debug: print column names


#########################
# Dashboard Layout / View
#########################
app = Dash(__name__)  # Initialize the JupyterDash application

#FIX ME Add in Grazioso Salvare’s logo
image_filename = 'Grazioso Salvare Logo.png'  # Define the correct logo file name with space
if not os.path.exists(image_filename):  # Check if file exists in current directory
    image_filename = os.path.join(os.path.expanduser('~'), 'Desktop', 'Grazioso Salvare Logo.png')  # Fallback to desktop path
try:
    encoded_image = base64.b64encode(open(image_filename, 'rb').read())  # Encode the logo image in base64
except FileNotFoundError:
    print(f"Optional logo: {image_filename} not found. Please ensure 'Grazioso Salvare Logo.png' is on the desktop or in the current directory.")  # Error message if file not found
    encoded_image = None  # Set to None to avoid crash if file is missing

#FIX ME Place the HTML image tag in the line below into the app.layout code according to your design
#FIX ME Also remember to include a unique identifier such as your name or date
#html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()))

app.layout = html.Div([  # Define the main dashboard layout
    html.Center(html.B(html.H1('Grazioso Salvare Rescue Dashboard - Developed by Karen Connors'))),  # Center the title with developer name
    html.A(html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()) if encoded_image else '', style={'height': '64px'}), href="http://www.snhu.edu") if encoded_image else html.P(
        "CS 499 Capstone | Database Enhancement | Synthetic Test Data",
        style={"color": "#555", "fontSize": "14px"}
    ),  # Add logo with link, fallback if missing
    html.Hr(),  # Horizontal rule for separation
    html.Div(  # Container for the dropdown filter
        dcc.Dropdown(
            id='filter-type',  # ID for callback reference
            options=[  # Define filter options
                {'label': 'Reset', 'value': 'all'},
                {'label': 'Water Rescue', 'value': 'water'},
                {'label': 'Mountain/Wilderness Rescue', 'value': 'mountain'},
                {'label': 'Disaster/Individual Tracking', 'value': 'disaster'}
            ],
            value='all',  # Default selection
            style={'width': '50%', 'marginBottom': '20px'}  # Styling for dropdown
        )
    ),
    html.Hr(),  # Horizontal rule for separation
    dash_table.DataTable(id='datatable-id',  # Interactive data table
                        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],  # Define columns
                        data=df.to_dict('records'),  # Initial data
                        page_size=10,  # Pagination limit
                        sort_action='native',  # Enable native sorting
                        style_table={'overflowX': 'auto', 'maxHeight': '300px'}  # Styling for table
                       ),
    html.Br(),  # Line break
    html.Hr(),  # Horizontal rule for separation
    #This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',  # Row container for side-by-side layout
             style={'display': 'flex'},  # Use flexbox for layout
             children=[  # Children divs for chart and map
        html.Div(
            id='graph-id',  # ID for pie chart
            className='col s12 m6',  # Bootstrap column class
        ),
        html.Div(
            id='map-id',  # ID for geolocation map
            className='col s12 m6',  # Bootstrap column class
        )
    ])
])

#############################################
# Interaction Between Components / Controller
#############################################

@app.callback(Output('datatable-id', 'data'),  # Output to update table data
              [Input('filter-type', 'value')])  # Input from dropdown
def update_dashboard(filter_type):
    """Retrieve rescue records using MongoDB-side filtering."""
    records = db.filterData(filter_type or "all")
    filtered_df = pd.DataFrame.from_records(records)

    # MongoDB ObjectId cannot be serialized by Dash.
    filtered_df.drop(
        columns=["_id"],
        inplace=True,
        errors="ignore"
    )

    return filtered_df.to_dict("records")


# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(  # Callback to update pie chart
    Output('graph-id', "children"),  # Output for chart children
    [Input('datatable-id', "derived_virtual_data")])  # Input from table data
def update_graphs(viewData):
    """Display breed distribution for currently filtered animals."""

    # On initial load, use the initial dashboard records.
    if viewData is None:
        chart_df = df.copy()
    else:
        chart_df = pd.DataFrame.from_records(viewData)

    # Handle empty results and missing breed information.
    if chart_df.empty or "breed" not in chart_df.columns:
        return [
            html.P(
                "No animal records available for this chart."
            )
        ]

    chart_df = chart_df.dropna(subset=["breed"])

    if chart_df.empty:
        return [
            html.P("No breed information available.")
        ]

    pie_fig = px.pie(
        chart_df,
        names="breed",
        title="Rescue Animal Breed Distribution"
    )

    pie_fig.update_layout(
        margin=dict(l=20, r=20, t=60, b=20)
    )

    return [dcc.Graph(figure=pie_fig)]


#This callback will highlight a cell on the data table when the user selects it
@app.callback(  # Callback to style selected columns
    Output('datatable-id', 'style_data_conditional'),  # Output for table styling
    [Input('datatable-id', 'selected_columns')])  # Input from selected columns
def update_styles(selected_columns):  # Function to apply styling
    # Fix: Handle None case for selected_columns
    if selected_columns is None:  # Check if no columns are selected
        return []  # Return empty list to avoid error
    return [{  # Return list of style conditions
        'if': {'column_id': i},  # Condition for specific column
        'background_color': '#D2F3FF'  # Highlight color
    } for i in selected_columns]  # Apply to each selected column

# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(  # Callback to update geolocation map
    Output('map-id', "children"),  # Output for map children
    [Input('datatable-id', "derived_virtual_data"),  # Input from table data
     Input('datatable-id', "derived_virtual_selected_rows")])  # Input from selected rows
def update_map(viewData, index):
    """Display the selected animal's location when coordinates exist."""

    default_center = [30.75, -97.48]

    def make_map(center, marker=None):
        children = [dl.TileLayer()]
        if marker is not None:
            children.append(marker)

        return dl.Map(
            style={"width": "100%", "height": "450px"},
            center=center,
            zoom=10,
            children=children
        )

    if not viewData:
        return [
            html.P("No animal records available for mapping."),
            make_map(default_center)
        ]

    selected_row = index[0] if index else 0

    if selected_row < 0 or selected_row >= len(viewData):
        return [
            html.P("Select a valid animal record."),
            make_map(default_center)
        ]

    animal = viewData[selected_row]

    # Accept either the original shelter schema or common
    # descriptive coordinate field names.
    latitude = animal.get(
        "location_lat",
        animal.get("latitude")
    )
    longitude = animal.get(
        "location_long",
        animal.get("longitude")
    )

    try:
        latitude = float(latitude)
        longitude = float(longitude)

        if not (
            -90 <= latitude <= 90
            and -180 <= longitude <= 180
        ):
            raise ValueError("Coordinates outside valid range")

    except (TypeError, ValueError, OverflowError):
        return [
            html.P(
                "Location coordinates are unavailable "
                "for the selected animal."
            ),
            make_map(default_center)
        ]

    # Exclude non-finite coordinate values.
    import math

    if not (
        math.isfinite(latitude)
        and math.isfinite(longitude)
    ):
        return [
            html.P("Valid location coordinates are unavailable."),
            make_map(default_center)
        ]

    position = [latitude, longitude]

    marker = dl.Marker(
        position=position,
        children=[
            dl.Tooltip(str(animal.get("breed", "Unknown breed"))),
            dl.Popup([
                html.H4("Animal Information"),
                html.P(
                    "Name: "
                    + str(animal.get("name", "Unknown"))
                ),
                html.P(
                    "Breed: "
                    + str(animal.get("breed", "Unknown"))
                )
            ])
        ]
    )

    return [make_map(position, marker)]


app.run(debug=False, port=8050, jupyter_mode="external")  # Start the Dash server in debug mode with a specific port to avoid conflicts

Error: /Users/karenconnors/Desktop/Grazioso Salvare Logo.png not found. Please ensure 'Grazioso Salvare Logo.png' is on the desktop or in the current directory.
Dash app running on http://127.0.0.1:8050/


In [2]:
import socket

print("Checking Dash connection on port 8050...")

try:
    with socket.create_connection(("127.0.0.1", 8050), timeout=3):
        print("PASS: Dash port 8050 is accepting connections.")
except OSError as error:
    print("FAIL: Dash port 8050 is not accepting connections.")
    print("Details:", error)

try:
    response = app.server.test_client().get("/")
    print("Dash application HTTP status:", response.status_code)
except Exception as error:
    print("Dash application test failed:", repr(error))

Checking Dash connection on port 8050...
PASS: Dash port 8050 is accepting connections.
Dash application HTTP status: 200
